In [18]:
"""
Static Security Benchmark
This script measures how well Bandit classifies vulnerable and patched code pairs from CSV input.
"""

import json
import os
import sys
import re
import time
import shutil
import subprocess
import tempfile
import pandas as pd
from datetime import datetime
from pathlib import Path
from llmScan import *

In [19]:
# Define benchmark test for one full code snippet using Bandit

def _bandit_valid_issues(issues):
    return [
        issue for issue in issues
        if str(issue.get("issue_confidence", "")).upper() == "HIGH"
        and str(issue.get("issue_severity", "")).upper() in ("MEDIUM", "HIGH")
    ]


_BANDIT_CHECKED = False
_BANDIT_COMMAND = None
_BANDIT_MISSING_LOGGED = False
_BANDIT_FORCE_MODULE = False


def _is_command_available(cmd):
    """Check whether the first executable in a command list is still valid."""
    if not cmd:
        return False
    exe = cmd[0]
    if os.path.isabs(exe):
        return os.path.exists(exe) and os.access(exe, os.X_OK)
    return shutil.which(exe) is not None


def _get_bandit_command():
    """Resolve Bandit command once: prefer binary, fallback to python -m bandit."""
    global _BANDIT_CHECKED, _BANDIT_COMMAND, _BANDIT_FORCE_MODULE
    if _BANDIT_CHECKED:
        # Refresh stale cached paths in long-lived notebook kernels.
        if _is_command_available(_BANDIT_COMMAND):
            return _BANDIT_COMMAND
        _BANDIT_CHECKED = False
        _BANDIT_COMMAND = None

    _BANDIT_CHECKED = True

    if not _BANDIT_FORCE_MODULE:
        bandit_bin = shutil.which("bandit")
        if bandit_bin:
            _BANDIT_COMMAND = [bandit_bin]
            return _BANDIT_COMMAND

    try:
        probe = subprocess.run(
            [sys.executable, "-m", "bandit", "--version"],
            check=False,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
        )
        if probe.returncode == 0:
            _BANDIT_COMMAND = [sys.executable, "-m", "bandit"]
            return _BANDIT_COMMAND
    except Exception:
        pass

    _BANDIT_COMMAND = None
    return _BANDIT_COMMAND


def _run_bandit(command):
    """Run Bandit and recover from broken launcher scripts once."""
    global _BANDIT_CHECKED, _BANDIT_COMMAND, _BANDIT_FORCE_MODULE
    try:
        return subprocess.run(
            command,
            check=False,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
        )
    except FileNotFoundError:
        # Common case: launcher exists but shebang points to missing interpreter.
        _BANDIT_CHECKED = False
        _BANDIT_COMMAND = None
        _BANDIT_FORCE_MODULE = True
        refreshed = _get_bandit_command()
        if not refreshed:
            raise
        retry_command = refreshed + command[len(command) - 5 :]
        return subprocess.run(
            retry_command,
            check=False,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
        )


def test_vulnerabilities_benchmark(python_code_text):
    """
    Analyze a full Python code snippet for security vulnerabilities using Bandit.

    Returns:
        (success(bool), is_vulnerable(bool|None), tool_name(str), scan_time(float))
    """
    start_time = time.time()
    tool_name = "Bandit"
    temp_path = None

    try:
        bandit_base = _get_bandit_command()
        if not bandit_base:
            global _BANDIT_MISSING_LOGGED
            if not _BANDIT_MISSING_LOGGED:
                print(
                    "Bandit is not available in this environment. "
                    "Install with: pip install bandit (or pip install -r requirements.txt)."
                )
                _BANDIT_MISSING_LOGGED = True
            scan_time = time.time() - start_time
            return False, None, tool_name, scan_time

        with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False, encoding="utf-8") as tmp_file:
            tmp_file.write(str(python_code_text or ""))
            temp_path = tmp_file.name

        command = bandit_base + ["-r", temp_path, "-ll", "-iii", "--format", "json"]
        proc = _run_bandit(command)

        global last_bandit_output
        last_bandit_output = (proc.stdout or "") + "\n" + (proc.stderr or "")

        data = json.loads(proc.stdout or "{}")
        all_issues = data.get("results", [])
        valid_issues = _bandit_valid_issues(all_issues)

        is_vulnerable = len(valid_issues) > 0
        scan_time = time.time() - start_time
        return True, is_vulnerable, tool_name, scan_time

    except Exception as e:
        last_bandit_output = locals().get("last_bandit_output", str(e))
        print(f"Bandit run failed: {e}\nLast output:\n{last_bandit_output[:1000]}")
        scan_time = time.time() - start_time
        return False, None, tool_name, scan_time

    finally:
        if temp_path and os.path.exists(temp_path):
            os.remove(temp_path)


def label_from_prediction(success, is_vulnerable):
    if not success or is_vulnerable is None:
        return "unknown"
    return "vulnerable" if is_vulnerable else "safe"


last_bandit_output = ""

print("test_vulnerabilities_benchmark function defined successfully!")

test_vulnerabilities_benchmark function defined successfully!


In [20]:
# Define CSV-driven benchmark runner
def run_benchmark_from_csv(
    csv_path,
    before_col="vulnerable_code",
    after_col="fixed_code",
    verbose=True,
):
    """
    Run Bandit benchmark from a CSV where each row contains a vulnerable/patched pair.

    Vulnerable and patched snippets are analyzed separately, each as a full snippet.
    """
    df = pd.read_csv(csv_path)

    required = [before_col, after_col]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required column(s): {missing}")

    output_df = df.copy()

    before_predictions = []
    after_predictions = []
    before_correct = []
    after_correct = []
    before_success = []
    after_success = []
    before_times = []
    after_times = []

    for idx, row in output_df.iterrows():
        before_code = row.get(before_col, "")
        after_code = row.get(after_col, "")

        b_success, b_pred, _, b_time = test_vulnerabilities_benchmark(before_code)
        a_success, a_pred, _, a_time = test_vulnerabilities_benchmark(after_code)

        b_label = label_from_prediction(b_success, b_pred)
        a_label = label_from_prediction(a_success, a_pred)

        before_predictions.append(b_label)
        after_predictions.append(a_label)
        before_success.append(b_success)
        after_success.append(a_success)
        before_times.append(round(b_time, 3))
        after_times.append(round(a_time, 3))

        expected_before = "vulnerable"
        expected_after = "safe"
        before_correct.append(b_label == expected_before)
        after_correct.append(a_label == expected_after)

        if verbose:
            print(
                f"Row {idx + 1}: before={b_label} (expected vulnerable), "
                f"after={a_label} (expected safe)"
            )

    output_df["expected_before"] = "vulnerable"
    output_df["expected_after"] = "safe"
    output_df["bandit_classification_before"] = before_predictions
    output_df["bandit_classification_after"] = after_predictions
    output_df["bandit_correct_before"] = before_correct
    output_df["bandit_correct_after"] = after_correct
    output_df["bandit_success_before"] = before_success
    output_df["bandit_success_after"] = after_success
    output_df["bandit_scan_time_before_seconds"] = before_times
    output_df["bandit_scan_time_after_seconds"] = after_times
    output_df["bandit_pair_correct"] = output_df["bandit_correct_before"] & output_df["bandit_correct_after"]

    return output_df

print("run_benchmark_from_csv function defined successfully!")

run_benchmark_from_csv function defined successfully!


In [21]:
# Run the CSV benchmark
print("Starting Bandit CSV Security Benchmark...")

# Update this path to your benchmark input CSV
input_csv_path = "sample2.csv"

bandit_cmd = _get_bandit_command()
if not bandit_cmd:
    raise RuntimeError(
        "Bandit is not installed in the current environment. "
        "Install with: pip install bandit (or pip install -r requirements.txt)."
    )

df_results = run_benchmark_from_csv(input_csv_path, verbose=False)

model_tag = "Bandit"
date_stamp = datetime.now().strftime("%Y%m%d")
output_file = f"benchmark{model_tag}{date_stamp}.csv"
df_results.to_csv(output_file, index=False)

print(f"Benchmark completed for {len(df_results)} rows.")
print(f"Results saved to: {output_file}")

Starting Bandit CSV Security Benchmark...


KeyboardInterrupt: 

In [ ]:
# Calculate and display pairwise metrics
print("=" * 60)
print("BANDIT CSV BENCHMARK RESULTS")
print("=" * 60)

total_rows = len(df_results)
before_acc = df_results["bandit_correct_before"].mean() if total_rows else 0
after_acc = df_results["bandit_correct_after"].mean() if total_rows else 0
pair_acc = df_results["bandit_pair_correct"].mean() if total_rows else 0

print(f"Total row pairs analyzed: {total_rows}")
print(f"Before-code accuracy (expected vulnerable): {before_acc:.2%}")
print(f"After-code accuracy (expected safe): {after_acc:.2%}")
print(f"Pair accuracy (both correct in same row): {pair_acc:.2%}")

before_unknown = (df_results["bandit_classification_before"] == "unknown").sum() if total_rows else 0
after_unknown = (df_results["bandit_classification_after"] == "unknown").sum() if total_rows else 0

print(f"Unknown classifications (before): {before_unknown}")
print(f"Unknown classifications (after): {after_unknown}")

avg_before_time = df_results["bandit_scan_time_before_seconds"].mean() if total_rows else 0
avg_after_time = df_results["bandit_scan_time_after_seconds"].mean() if total_rows else 0
print(f"Average scan time before-code: {avg_before_time:.3f}s")
print(f"Average scan time after-code: {avg_after_time:.3f}s")

BANDIT CSV BENCHMARK RESULTS
Total row pairs analyzed: 200
Before-code accuracy (expected vulnerable): 22.00%
After-code accuracy (expected safe): 87.00%
Pair accuracy (both correct in same row): 9.00%
Unknown classifications (before): 0
Unknown classifications (after): 0
Average scan time before-code: 0.242s
Average scan time after-code: 0.239s


In [ ]:
# Optional compact summary export
summary = {
    "total_row_pairs": len(df_results),
    "before_accuracy": float(df_results["bandit_correct_before"].mean()) if len(df_results) else 0.0,
    "after_accuracy": float(df_results["bandit_correct_after"].mean()) if len(df_results) else 0.0,
    "pair_accuracy": float(df_results["bandit_pair_correct"].mean()) if len(df_results) else 0.0,
    "before_success_rate": float(df_results["bandit_success_before"].mean()) if len(df_results) else 0.0,
    "after_success_rate": float(df_results["bandit_success_after"].mean()) if len(df_results) else 0.0,
    "unknown_before": int((df_results["bandit_classification_before"] == "unknown").sum()) if len(df_results) else 0,
    "unknown_after": int((df_results["bandit_classification_after"] == "unknown").sum()) if len(df_results) else 0,
}

summary_df = pd.DataFrame([summary])
summary_file = "bandit_csv_benchmark_summary.csv"
summary_df.to_csv(summary_file, index=False)
print(f"Summary metrics saved to: {summary_file}")

Summary metrics saved to: bandit_csv_benchmark_summary.csv


In [ ]:
# Debug Bandit Input/Output from CSV rows
def debug_bandit_from_csv(csv_path, row_index=0, before_col="vulnerable_code", after_col="fixed_code"):
    df = pd.read_csv(csv_path)
    if df.empty:
        print("CSV is empty.")
        return
    if row_index < 0 or row_index >= len(df):
        print(f"row_index {row_index} is out of range (0-{len(df)-1})")
        return

    row = df.iloc[row_index]
    before_code = row.get(before_col, "")
    after_code = row.get(after_col, "")

    print("=" * 60)
    print(f"BANDIT CSV DEBUG | Row {row_index + 1}")
    print("=" * 60)

    b_success, b_pred, _, b_time = test_vulnerabilities_benchmark(before_code)
    a_success, a_pred, _, a_time = test_vulnerabilities_benchmark(after_code)

    print(f"Before expected: vulnerable | predicted: {label_from_prediction(b_success, b_pred)} | success: {b_success} | time: {b_time:.3f}s")
    print(f"After expected: safe       | predicted: {label_from_prediction(a_success, a_pred)} | success: {a_success} | time: {a_time:.3f}s")

    print("\nLast Bandit raw output (first 1000 chars):")
    print(last_bandit_output[:1000])

# Example debug usage
# debug_bandit_from_csv(input_csv_path, row_index=0)